In [ ]:
!nvidia-smi

Wed Feb 18 18:53:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L40                     Off |   00000000:00:10.0 Off |                    0 |
| N/A   61C    P0             90W /  300W |   16070MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
import ollama
import pickle
import os

# --- CONFIGURATION ---
FILE_PATH = 'contents_only_oran-1.txt'
DB_FILE = 'vector_db.pkl'
EMBEDDING_MODEL = 'hf.co/CompendiumLabs/bge-base-en-v1.5-gguf'
LANGUAGE_MODEL = 'hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF'
CHUNK_SIZE = 1000  
CHUNK_OVERLAP = 100 

def create_chunks(text, size, overlap):
    chunks = []
    for i in range(0, len(text), size - overlap):
        chunk_content = text[i:i + size]
        # Adding metadata: track the character offset for reference
        metadata = {"source": FILE_PATH, "start_char": i}
        chunks.append({"content": chunk_content, "metadata": metadata})
    return chunks

def cosine_similarity(a, b):
    dot_product = sum([x * y for x, y in zip(a, b)])
    norm_a = sum([x ** 2 for x in a]) ** 0.5
    norm_b = sum([x ** 2 for x in b]) ** 0.5
    return dot_product / (norm_a * norm_b)

# --- 1. LOAD OR BUILD DATABASE ---
VECTOR_DB = []

if os.path.exists(DB_FILE):
    print(f"--- Loading existing database from {DB_FILE} ---")
    with open(DB_FILE, 'rb') as f:
        VECTOR_DB = pickle.load(f)
    print(f"Loaded {len(VECTOR_DB)} chunks.")
else:
    print(f"--- Database not found. Processing {FILE_PATH} ---")
    with open(FILE_PATH, 'r', encoding='utf-8') as file:
        raw_text = file.read()
    
    raw_chunks = create_chunks(raw_text, CHUNK_SIZE, CHUNK_OVERLAP)
    
    for i, item in enumerate(raw_chunks):
        try:
            # truncate=True ensures we never hit the error you saw earlier
            response = ollama.embed(model=EMBEDDING_MODEL, input=item['content'], truncate=True)
            embedding = response['embeddings'][0]
            
            # Store as a dictionary for easier access
            VECTOR_DB.append({
                "content": item['content'],
                "metadata": item['metadata'],
                "embedding": embedding
            })
            
            if (i + 1) % 10 == 0 or (i + 1) == len(raw_chunks):
                print(f"Embedded {i+1}/{len(raw_chunks)} chunks...")
        except Exception as e:
            print(f"Error at chunk {i}: {e}")

    # Save to disk
    with open(DB_FILE, 'wb') as f:
        pickle.dump(VECTOR_DB, f)
    print("Database saved successfully!")

# --- 2. RETRIEVAL LOGIC ---
def retrieve(query, top_n=3):
    query_emb = ollama.embed(model=EMBEDDING_MODEL, input=query)['embeddings'][0]
    similarities = []
    for item in VECTOR_DB:
        score = cosine_similarity(query_emb, item['embedding'])
        similarities.append((item, score))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_n]

# --- 3. CHAT INTERFACE ---
input_query = input('\nAsk me a question: ')
retrieved_results = retrieve(input_query)

print('\n--- Context Found ---')
context_text = ""
for item, score in retrieved_results:
    print(f" - [Score: {score:.2f}] (Offset: {item['metadata']['start_char']})")
    context_text += f"\n- {item['content']}"

instruction_prompt = f'''You are a helpful chatbot.
Use only the following pieces of context to answer the question. 
If the answer isn't in the context, say you don't know.

Context:
{context_text}
'''

print('\nChatbot response:')
stream = ollama.chat(
    model=LANGUAGE_MODEL,
    messages=[
        {'role': 'system', 'content': instruction_prompt},
        {'role': 'user', 'content': input_query},
    ],
    stream=True,
)

for chunk in stream:
    print(chunk['message']['content'], end='', flush=True)
print("\n")



--- Loading existing database from vector_db.pkl ---
Loaded 12614 chunks.


IndexError: list index out of range

In [3]:
import ollama
import pickle
import os
import time  # New import for timing

# --- CONFIGURATION ---
FILE_PATH = 'contents_only_oran-1.txt'
DB_FILE = 'vector_db.pkl'
EMBEDDING_MODEL = 'hf.co/CompendiumLabs/bge-base-en-v1.5-gguf'
LANGUAGE_MODEL = 'hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF'
CHUNK_SIZE = 1000  
CHUNK_OVERLAP = 100 

def create_chunks(text, size, overlap):
    chunks = []
    for i in range(0, len(text), size - overlap):
        chunk_content = text[i:i + size]
        metadata = {"source": FILE_PATH, "start_char": i}
        chunks.append({"content": chunk_content, "metadata": metadata})
    return chunks

def cosine_similarity(a, b):
    dot_product = sum([x * y for x, y in zip(a, b)])
    norm_a = sum([x ** 2 for x in a]) ** 0.5
    norm_b = sum([x ** 2 for x in b]) ** 0.5
    return dot_product / (norm_a * norm_b)

# --- 1. LOAD OR BUILD DATABASE (WITH LATENCY LOGGING) ---
VECTOR_DB = []
db_start_time = time.perf_counter()

if os.path.exists(DB_FILE):
    print(f"--- Loading existing database from {DB_FILE} ---")
    with open(DB_FILE, 'rb') as f:
        VECTOR_DB = pickle.load(f)
    db_end_time = time.perf_counter()
    print(f"✅ Loaded {len(VECTOR_DB)} chunks in {db_end_time - db_start_time:.4f} seconds.")
else:
    print(f"--- Database not found. Processing {FILE_PATH} ---")
    with open(FILE_PATH, 'r', encoding='utf-8') as file:
        raw_text = file.read()
    
    raw_chunks = create_chunks(raw_text, CHUNK_SIZE, CHUNK_OVERLAP)
    
    for i, item in enumerate(raw_chunks):
        try:
            response = ollama.embed(model=EMBEDDING_MODEL, input=item['content'], truncate=True)
            embedding = response['embeddings'][0]
            VECTOR_DB.append({
                "content": item['content'],
                "metadata": item['metadata'],
                "embedding": embedding
            })
        except Exception as e:
            print(f"Error at chunk {i}: {e}")

    with open(DB_FILE, 'wb') as f:
        pickle.dump(VECTOR_DB, f)
    
    db_end_time = time.perf_counter()
    print(f"✅ Created and saved database in {db_end_time - db_start_time:.4f} seconds.")

# --- 2. RETRIEVAL (WITH LATENCY LOGGING) ---
def retrieve(query, top_n=3):
    start_retrieval = time.perf_counter()
    
    query_emb = ollama.embed(model=EMBEDDING_MODEL, input=query)['embeddings'][0]
    similarities = []
    for item in VECTOR_DB:
        score = cosine_similarity(query_emb, item['embedding'])
        similarities.append((item, score))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    results = similarities[:top_n]
    
    end_retrieval = time.perf_counter()
    print(f"🔍 Retrieval Phase took: {end_retrieval - start_retrieval:.4f} seconds.")
    return results

# --- 3. GENERATION (WITH LATENCY LOGGING) ---
input_query = input('\nAsk me a question: ')
retrieved_results = retrieve(input_query)

context_text = "\n".join([f"- {item['content']}" for item, score in retrieved_results])

instruction_prompt = f'''You are a helpful chatbot. Use the context below to answer.
Context:
{context_text}
'''

print('\n--- Context Found ---')
context_text = ""
for item, score in retrieved_results:
    print(f" - [Score: {score:.2f}] (Offset: {item['metadata']['start_char']})")
    print(item)
    context_text += f"\n- {item['content']}"
    

print('\nChatbot response:')
start_gen = time.perf_counter()
first_token_received = False

stream = ollama.chat(
    model=LANGUAGE_MODEL,
    messages=[
        {'role': 'system', 'content': instruction_prompt},
        {'role': 'user', 'content': input_query},
    ],
    stream=True,
)

for chunk in stream:
    if not first_token_received:
        # Measure time to first token (TTFT)
        ttft = time.perf_counter() - start_gen
        first_token_received = True
    print(chunk['message']['content'], end='', flush=True)

end_gen = time.perf_counter()
print(f"\n\n🤖 Generation Phase took: {end_gen - start_gen:.4f} seconds.")
print(f"⚡ Time to First Token (TTFT): {ttft:.4f} seconds.")



--- Loading existing database from vector_db.pkl ---
✅ Loaded 12614 chunks in 0.4818 seconds.
🔍 Retrieval Phase took: 1.8529 seconds.

--- Context Found ---
 - [Score: 0.65] (Offset: 6744600)
{'content': 'Tools One of the key objectives of interoperability testing is to validate the functionality of production grade DUTs. Hence it is important to ensure that the DUTs are not negatively impacted with the utilization of internal functions solely to support interoperability testing. ie, DUTs are not expected to be testing tools when deployed in production networks and therefore DUTs should not be used as testing tools during interoperability tests. Interoperability tests are performed with a set of testing tools which are used to both apply active stimulus and as well as passive monitoring and measurements of the DUTs.\n\nThe test is applicable to deployment scenarios where the O-CU and O-DU are implemented as disaggregated nodes connected by an F1 interface and scenarios where they are i